<a href="https://colab.research.google.com/github/adljna/ProjectA-Group3-KematianAliKhamenei/blob/main/Content%20Scraping%20%26%20Preprocessing/Antara/2_Antara_ContentScrapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Content Scrapping Indonesian Text

**(1) Instalasi Library**

Bagian ini berisi instalasi semua library Python yang dibutuhkan untuk scraping

In [2]:
# Library untuk scraping dan manipulasi data
!pip install beautifulsoup4 requests pandas

**(2) Impor Library**

Bagian ini mengimpor semua modul dan fungsi yang diperlukan dari library yang telah diinstal. Import dikelompokkan berdasarkan fungsinya.

In [3]:
# Data Handling
import pandas as pd
import numpy as np

# Web Scraping
import requests
from bs4 import BeautifulSoup
from google.colab import files

**(3) Data Scraping**

Bagian ini bertanggung jawab untuk mengunggah file CSV yang berisi tautan (links) dan melakukan scraping konten artikel dari tautan tersebut. Menggunakan `requests` dan `BeautifulSoup`.

In [4]:
from google.colab import files

# Upload the CSV file
uploaded = files.upload()
filename = list(uploaded.keys())[0]

Saving Tugas 1A Kelompok 3 - Antara.csv to Tugas 1A Kelompok 3 - Antara.csv


In [5]:
import io
try:
    # Modified: Changed from pd.read_excel to pd.read_csv and removed sheet_name
    links_df = pd.read_csv(io.BytesIO(uploaded[filename]))

    if 'Link' in links_df.columns:
        urls = links_df['Link'].tolist()
        print(f"Successfully loaded {len(urls)} URLs from '{filename}'.")
    else:
        print("Error: 'link' column not found in the uploaded file. Please check your CSV file.")
        urls = [] # Set urls to empty if column not found
except FileNotFoundError:
    print("Error: Uploaded file not found. Please upload the CSV file.")
    urls = [] # Set urls to empty if file not found
except Exception as e:
    print(f"An error occurred while reading the file: {e}")
    urls = []
# --------------------------------------------------

# Function to scrape content from a URL
def scrape_content(url):
    try:
        response = requests.get(url)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')

            # Extract the title of the article
            title = soup.find('title').text

            # Extract the article's content
            paragraphs = soup.find_all('p')
            content = "\n".join([para.text for para in paragraphs])

            return {
                "url": url,
                "title": title,
                "content": content
            }

        else:
            return {
                "url": url,
                "title": None,
                "content": None,
                "error": f"Failed to fetch page, status code: {response.status_code}"
            }
    except Exception as e:
        return {
            "url": url,
            "title": None,
            "content": None,
            "error": str(e)
        }

# Scrape each URL and store the results in a list of dictionaries
data = []
if urls: # Only proceed if URLs were successfully loaded
    for url in urls:
        result = scrape_content(url)
        data.append(result)
else:
    print("No URLs to scrape. Please check your uploaded CSV file and ensure it has a 'link' column.")

# Create a pandas DataFrame from the list of dictionaries
df_article = pd.DataFrame(data)

# Display the DataFrame
df_article.head()

# Save the DataFrame to a CSV file (optional)
df_article.to_csv('scraped_articles.csv', index=False)

Successfully loaded 100 URLs from 'Tugas 1A Kelompok 3 - Antara.csv'.


In [6]:
# Function to scrape content from a URL
def scrape_content(url):
    try:
        # Add a User-Agent header to mimic a browser and avoid 403 Forbidden errors
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')

            # Extract the title of the article
            title = soup.find('title').text if soup.find('title') else 'No Title Found'

            # Extract the article's content
            # Try to find common article body tags first
            article_content_div = soup.find('article') or soup.find('div', class_='article-content') or soup.find('div', class_='detail_text')

            if article_content_div:
                paragraphs = article_content_div.find_all('p')
            else:
                paragraphs = soup.find_all('p')

            content = "\n".join([para.text for para in paragraphs])

            return {
                "url": url,
                "title": title,
                "content": content
            }

        else:
            return {
                "url": url,
                "title": None,
                "content": None,
                "error": f"Failed to fetch page, status code: {response.status_code}"
            }
    except Exception as e:
        return {
            "url": url,
            "title": None,
            "content": None,
            "error": str(e)
        }

**(4) Ekspor Hasil**

Bagian ini menyimpan DataFrame akhir yang berisi semua data hasil scrapping ke dalam file CSV.

In [7]:
df_article.to_csv(r'content_articles.csv', index=False)
print("Hasil analisis disimpan ke 'content_articles.csv'")

Hasil analisis disimpan ke 'content_articles.csv'
